# Comparaison d'algos en Classification Supervisée

Nous allons comparer les différentes méthodes de classification supervisée n que nous avons vues jusqu'à présent, la régression logistique avec toutes les variables, puis  avec les variables sélectionnées par le critère BIC avec un algo backward et par AIC puis les méthodes de vraisemblance pénalisée.
Le nombre de bloc de la validation croisée vaut $k=10$ mais peut être modifié par l'utilisateur. Les données s'appellent *don* et la variable d'intérêt $Y$

In [58]:
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
import sklearn.metrics as sklm
from patsy import dmatrix

import logistic_step_sk as lss 

In [59]:
don = pd.read_csv("dfbase.csv",header=0,sep=",")
don.head(3)

,famhist[T.Present],adiposity,age,alcohol,ldl,obesity,sbp,tobacco,typea,Y
0,1.0,23.11,52.0,97.20,5.73,25.30,160.0,12.00,49.0,1
1,0.0,28.61,63.0,2.06,4.41,28.87,144.0,0.01,55.0,1
2,1.0,32.28,46.0,3.81,3.48,29.14,118.0,0.08,52.0,0


Codage des variables qualitatives famhist

In [60]:
X = don.drop(columns=["Y"])
X = X.to_numpy()
Y = don["Y"].to_numpy()

In [61]:
nb=10
skf = StratifiedKFold(n_splits=nb, shuffle=True, random_state=123)
PROB = pd.DataFrame({"Y":Y,"log":0.0,"BIC":0.0,"AIC":0.0,
                    "ridge":0.0,"lasso":0.0,"elast":0.0,"arbre":0.0,"foret":0.0})

choix des grilles de régularisation

In [62]:
def grille(X, y, type = "lasso", ng=100):
    scalerX = StandardScaler().fit(X)
    Xcr= scalerX.transform(X)
    l0 = np.abs(Xcr.transpose().dot((y-y.mean()))).max()/X.shape[0]
    llc = np.linspace(0,-4,ng)
    ll = l0*10**llc
    if type=="lasso":
        Cs = 1/ 0.9/ X.shape[0] / (l0*10**(llc))
    elif type=="ridge":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 100)
    elif type=="enet":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 2)
    return Cs

On compare les méthodes en utilisant skf

In [67]:
for app_index, val_index in skf.split(X,Y):
    Xapp = X[app_index,:]
    Xtest = X[val_index,:]
    Yapp = Y[app_index]
    ### logistique
    log = LogisticRegression(penalty=None,solver="newton-cholesky").fit(Xapp,Yapp)
    PROB.loc[val_index,"log"] = log.predict_proba(Xtest)[:,1]
    ### bic
    choixbic = lss.LogisticRegressionSelectionFeatureIC(start=[],direction="forward",crit="bic",multi_class='auto').fit(Xapp,Yapp)
    PROB.loc[val_index, "BIC"] = choixbic.predict_proba(Xtest)[:,1]
    ### aic
    choixaic = lss.LogisticRegressionSelectionFeatureIC(start=[],direction="forward",crit="aic",multi_class='auto').fit(Xapp,Yapp)
    PROB.loc[val_index, "AIC"] = choixaic.predict_proba(Xtest)[:,1]
    ### lasso
    cr = StandardScaler()
    Cs_lasso = grille(Xapp,Yapp, "lasso")
    lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10,Cs=Cs_lasso,  solver="saga", max_iter=2000)
    pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
    pipe_lassocv.fit(Xapp,Yapp)
    PROB.loc[val_index,"lasso"] = pipe_lassocv.predict_proba(Xtest)[:,1]
    ### elastic net
    cr = StandardScaler()
    Cs_enet = grille(Xapp,Yapp,"enet")
    enetcv=LogisticRegressionCV(cv=10,penalty="elasticnet",n_jobs=10,l1_ratios=[0.5],Cs=Cs_enet,solver="saga",max_iter=2000)
    pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
    pipe_enetcv.fit(Xapp,Yapp)
    PROB.loc[val_index,"elast"] = pipe_enetcv.predict_proba(Xtest)[:,1] 
    ### ridge
    cr = StandardScaler()
    Cs_ridge = grille(Xapp,Yapp,"ridge")
    ridgecv = LogisticRegressionCV(cv=10, penalty="l2",Cs=Cs_ridge,  max_iter=1000)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PROB.loc[val_index,"ridge"] = pipe_ridgecv.predict_proba(Xtest)[:,1]
    ###arbre
    arbre = DecisionTreeClassifier(min_samples_leaf=5).fit(Xapp,Yapp)
    PROB.loc[val_index,"arbre"] = arbre.predict_proba(Xtest)[:,1]
    ###foret
    foret = RandomForestClassifier().fit(Xapp,Yapp)
    PROB.loc[val_index,"foret"] = foret.predict_proba(Xtest)[:,1]

In [68]:
round(PROB.iloc[0:4,:],3)

,Y,log,BIC,AIC,ridge,lasso,elast,arbre,foret
0,1,0.742,0.689,0.689,0.677,0.598,0.605,1.000,0.83
1,1,0.292,0.362,0.333,0.321,0.373,0.358,0.000,0.31
2,0,0.251,0.313,0.230,0.286,0.314,0.297,0.667,0.11
3,1,0.719,0.720,0.681,0.679,0.696,0.682,1.000,0.68


In [69]:
PROB.to_csv("PROB.csv",index=False)

In [29]:
PROB.to_csv("PROBpoly.csv",index=False)

In [36]:
PROB.to_csv("PROBinter.csv",index=False)